In [ ]:
import os, time, requests, pandas as pd
from dataclasses import dataclass, asdict
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- site config --------------------
SITE_ROOT = "https://expinterweb.mites.gob.es"
BASE_URL = f"{SITE_ROOT}/regcon/pub/consultaPublicaEstatal"

def make_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    
    # Anti-bot mitigations for MITES
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    
    # Execute CDP command to hide webdriver flag from the site's JS
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    return driver

# -------------------- small utils --------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def append_rows(out_dir, rows):
    """Append rows to CSV immediately to prevent data loss."""
    csv_path = os.path.join(out_dir, "spain_agreements.csv")
    df = pd.DataFrame([asdict(r) if hasattr(r, "__dict__") else r for r in rows])
    mode, header = ("a", False) if os.path.exists(csv_path) else ("w", True)
    df.to_csv(csv_path, mode=mode, header=header, index=False, encoding="utf-8")

# -------------------- parsing --------------------
@dataclass
class AgreementSpain:
    codigo: str | None
    denominacion: str | None
    tipo_tramite: str | None
    autoridad_laboral: str | None
    vigencia_desde: str | None
    vigencia_hasta: str | None
    detalle_url: str | None
    pdf_url: str | None
    pdf_saved_as: str | None

def download_pdf(url, out_dir, agreement_id):
    """Downloads the PDF and returns the local filename. Includes User-Agent to avoid blocks."""
    if not url: return None
    ensure_dir(out_dir)
    
    # Using the agreement ID as the filename to securely map metadata to the PDF
    name = f"{agreement_id}.pdf" 
    path = os.path.join(out_dir, name)
    
    if os.path.exists(path):   
        return path # skip re-download
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        r = requests.get(url, headers=headers, timeout=60)
        r.raise_for_status()
        if "application/pdf" in r.headers.get("Content-Type", "") or r.content.startswith(b"%PDF"):
            with open(path, "wb") as f:
                f.write(r.content)
            return path
        else:
            print(f"Warning: URL did not return a valid PDF: {url}")
            return None
    except Exception as e:
        print(f"Failed to download PDF {url}: {e}")
        return None

# -------------------- main scraper --------------------
def scrape_spain(out_dir="spain_data", target_count=10, pause=2.0):
    ensure_dir(out_dir)
    pdf_dir = os.path.join(out_dir, "pdfs")
    ensure_dir(pdf_dir)

    d = make_driver(headless=False) # Keep False to watch it bypass the error
    collected_agreements = []

    # Define the date ranges to search (Format: DD/MM/YYYY)
    # We will just use one month for the prototype test
    date_ranges = [
        ("01/01/2023", "31/01/2023")
    ]

    try:
        for date_from, date_to in date_ranges:
            print(f"\n--- Loading {BASE_URL} ---")
            d.get(BASE_URL)
            time.sleep(3) # Let the page load
            
            print(f"Entering date filter: {date_from} to {date_to}...")
            
            # 1. Find the "Desde" (From) date box and type the date
            box_desde = WebDriverWait(d, 10).until(
                EC.presence_of_element_located((By.XPATH, "//input[contains(translate(@id, 'DESDE', 'desde'), 'desde') or contains(translate(@name, 'DESDE', 'desde'), 'desde')]"))
            )
            box_desde.clear()
            box_desde.send_keys(date_from)

            # 2. Find the "Hasta" (To) date box and type the date
            box_hasta = d.find_element(By.XPATH, "//input[contains(translate(@id, 'HASTA', 'hasta'), 'hasta') or contains(translate(@name, 'HASTA', 'hasta'), 'hasta')]")
            box_hasta.clear()
            box_hasta.send_keys(date_to)

            # 3. Click the Search ("Buscar") button
            print("Clicking search button...")
            search_btn = d.find_element(By.XPATH, "//input[@type='submit' or @value='Buscar'] | //button[contains(translate(text(), 'BUSCAR', 'buscar'), 'buscar')]")
            d.execute_script("arguments[0].click();", search_btn)

            # 4. Wait for the results table to populate
            print("Waiting for the results table to load...")
            try:
                WebDriverWait(d, 15).until(EC.presence_of_element_located((By.XPATH, "//table//tr/td")))
                print("Table loaded successfully!")
            except:
                print("Timeout waiting for table. The site might be slow or found no results for these dates.")
                continue # Skip to the next date range if this one fails
            
            # 5. Parse the table rows
            rows = d.find_elements(By.XPATH, "//table//tr[td]")
            print(f"Found {len(rows)} agreements for this date range.")
            
            for row in rows[:target_count]:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) < 7: continue
                
                codigo = cols[0].text.strip()
                
                detalle_link = None
                try:
                    # Look for the detail link in the last column
                    action_a = cols[-1].find_element(By.TAG_NAME, "a")
                    detalle_link = action_a.get_attribute("href")
                except:
                    pass

                ag = AgreementSpain(
                    codigo=codigo,
                    denominacion=cols[1].text.strip(),
                    tipo_tramite=cols[2].text.strip(),
                    autoridad_laboral=cols[3].text.strip(),
                    vigencia_desde=cols[4].text.strip(),
                    vigencia_hasta=cols[5].text.strip(),
                    detalle_url=detalle_link,
                    pdf_url=None,
                    pdf_saved_as=None
                )
                collected_agreements.append(ag)

            # 6. Visit each detail page to extract the actual PDF link
            for ag in collected_agreements:
                if not ag.detalle_url:
                    continue
                    
                print(f"Fetching PDF details for agreement {ag.codigo}...")
                d.get(ag.detalle_url)
                time.sleep(pause) # Crucial for not getting IP banned
                
                try:
                    # Look for a link ending in .pdf or containing the word 'Descargar'/'PDF'
                    pdf_a = d.find_element(By.XPATH, "//a[contains(@href, '.pdf') or contains(translate(text(), 'PDF', 'pdf'), 'pdf')]")
                    ag.pdf_url = pdf_a.get_attribute("href")
                    
                    # Download and map PDF
                    if ag.pdf_url:
                        ag.pdf_saved_as = download_pdf(ag.pdf_url, pdf_dir, ag.codigo)
                except Exception as e:
                    print(f"Could not find PDF link for {ag.codigo}")
                
                # Save progress immediately
                append_rows(out_dir, [ag])

            # Stop the loop if we hit our target count across all dates
            if len(collected_agreements) >= target_count:
                break

        print(f"\nSuccessfully processed {len(collected_agreements)} agreements for prototype.")
        
    finally:
        d.quit()

    # Return summary dataframe
    csv_path = os.path.join(out_dir, "spain_agreements.csv")
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()

# To run the prototype:
df = scrape_spain(target_count=10)
print(df.head())

Loading https://expinterweb.mites.gob.es/regcon/pub/consultaPublicaEstatal...
